# 02 - Pull Wikidata

Lists the film and person ids found in the TMDB shards, then asks Wikidata about each. Run it after notebook 01 has finished every chunk: the id lists refuse to build while the pull is unfinished. The shards are uploaded to the private Hub repo `Tsomaros/llm-movies-raw` every 30 minutes during the run and at the end.
Use a **CPU runtime**. Needs the Colab Secret `HF_TOKEN` (write access); `TMDB_READ_TOKEN` is read for symmetry and not used here.

**To resume: rerun this notebook.** It downloads the shards from the repo and skips the ids already in them. A dead session loses at most the last 30 minutes.
The last cell is **one chain** (`&&`): if a download or any step fails, the rest do not run, so a Wikidata pull can never start on a failed download and overwrite the repo's shard.

**`PARTIAL`** is for a sample run only (notebook 01 with `LIMIT` set): it lets the id lists be built from an unfinished pull. Leave it off for the real run.

## Setup

Run these cells top to bottom. They are the same in all three notebooks.

In [ ]:
# clone the repo the first time, update it after that
!test -d /content/LLM_Movies_Model/.git && git -C /content/LLM_Movies_Model pull || git clone https://github.com/Tsomaros/LLM_Movies_Model /content/LLM_Movies_Model
%cd /content/LLM_Movies_Model

In [ ]:
!pip install -q uv
!uv sync --frozen

## Secrets and environment

Reads `TMDB_READ_TOKEN` and `HF_TOKEN` (write access) from Colab Secrets, with notebook access switched on. They are never printed. `UV_FROZEN` stops later `uv run` calls from re-resolving the environment.

In [ ]:
import os

from google.colab import userdata

os.environ["TMDB_READ_TOKEN"] = userdata.get("TMDB_READ_TOKEN")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["UV_FROZEN"] = "1"

## Sample switch

Off for the real run.

In [ ]:
PARTIAL = False  #@param {type:"boolean"}
PARTIAL_FLAG = "--partial" if PARTIAL else ""

## Download, list ids, pull films, pull people

In [ ]:
!uv run python -m llm_movies.data.hub_store download --dir /content/raw/tmdb --path-in-repo tmdb && uv run python -m llm_movies.data.hub_store download --dir /content/raw/wd_films --path-in-repo wd_films && uv run python -m llm_movies.data.hub_store download --dir /content/raw/wd_people --path-in-repo wd_people && uv run python -m llm_movies.data.shard_ids films --tmdb /content/raw/tmdb --out /content/raw/ids/film_ids.txt {PARTIAL_FLAG} && uv run python -m llm_movies.data.shard_ids people --tmdb /content/raw/tmdb --out /content/raw/ids/person_ids.txt {PARTIAL_FLAG} && uv run python -m llm_movies.data.wikidata films --ids /content/raw/ids/film_ids.txt --out /content/raw/wd_films --repo Tsomaros/llm-movies-raw && uv run python -m llm_movies.data.wikidata people --ids /content/raw/ids/person_ids.txt --out /content/raw/wd_people --repo Tsomaros/llm-movies-raw